In [ ]:
from faker import Faker
from random import choice, choices, randint, triangular, uniform
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from unicodedata import normalize
from re import sub, IGNORECASE

In [ ]:
load_dotenv()

DB_URL = os.getenv("DB_URL")
engine = create_engine(DB_URL)
faker = Faker('pt_BR')

In [ ]:
def apenas_numeros(valor: str) -> str:
    return sub(r'\D', '', valor)

def remove_acentos(texto: str) -> str:
    return normalize('NFKD', texto.lower()).encode('ASCII', 'ignore').decode('ASCII')

In [ ]:
# Constantes
QTD_LOJAS               = 10
QTD_BANCOS              = 5
QTD_CONTAS              = 120
QTD_PAGAMENTOS          = 1000
QTD_VENDAS              = 10000
QTD_TRANSPORTADORAS     = 8
QTD_FUNCIONARIOS        = 50
QTD_CONTA_LOJA          = 15
QTD_CONTA_FUNCIONARIO   = 60
QTD_PAGAMENTO_PARCELADO = 300

In [ ]:
dados_loja = []
for i in range(QTD_LOJAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_loja.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'email': faker.email(),
        'telefone': telefone_formatado,
        'percentual_comissao': round(uniform(0.01, 0.2), 2)
    })

df_lojas = pd.DataFrame(dados_loja)
display(df_lojas.head())

In [ ]:
dados_banco = []
for i in range(QTD_BANCOS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    
    dados_banco.append({
        'nome': f'Banco {faker.company()}',
        'cnpj': cnpj_formatado
    })

df_bancos = pd.DataFrame(dados_banco)
display(df_bancos.head())

In [ ]:
dados_conta = []

TIPOS_CONTA = ['CORRENTE', 'POUPANCA', 'SALARIO']

for i in range(QTD_CONTAS):
    dados_conta.append({
        'agencia': str(randint(1, 9999)).zfill(4),
        'numero_conta': str(randint(1, 999999)).zfill(6),
        'tipo_conta': choice(TIPOS_CONTA),
        'saldo': round(triangular(0, 9999.99, 100), 2),
        'cod_banco': randint(1, QTD_BANCOS)
    })

df_contas = pd.DataFrame(dados_conta)
display(df_contas.head())

In [ ]:
dados_pagamento = []

TIPOS_PAGAMENTO = ['BOLETO', 'PIX', 'DEBITO', 'CREDITO']
STATUS_PAGAMENTO = ['AGUARDANDO', 'CONCLUIDO', 'EXPIRADO']

for i in range(QTD_PAGAMENTOS):
    cod_conta_origem = randint(1, QTD_CONTAS)
    cod_conta_destino = randint(1, QTD_CONTAS)
    while cod_conta_destino == cod_conta_origem:
        cod_conta_destino = randint(1, QTD_CONTAS)
    
    dados_pagamento.append({
        'tipo_pagamento': choice(TIPOS_PAGAMENTO),
        'status_pagamento': choice(STATUS_PAGAMENTO),
        'valor': round(triangular(0, 999.99, 100), 2),
        'cod_conta_origem': cod_conta_origem,
        'cod_conta_destino': cod_conta_destino
    })

df_pagamentos = pd.DataFrame(dados_pagamento)
display(df_pagamentos.head())

In [ ]:
dados_funcionario = []
for i in range(QTD_FUNCIONARIOS):
    nome = sub(r'\b(sr|sra|srta|dr|dra)\.\s*', '', faker.name(), flags=IGNORECASE)
    cpf_formatado = apenas_numeros(faker.cpf())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    email = f'{sub(r' ', '_', remove_acentos(nome))}@gmail.com'
    
    dados_funcionario.append({
        'nome': nome,
        'cpf': cpf_formatado,
        'cargo': 'Vendedor',
        'salario': round(triangular(1621, 4000, 100), 2),
        'email': email,
        'telefone': telefone_formatado,
        'cod_loja': randint(1, QTD_LOJAS)
    })

df_funcionarios = pd.DataFrame(dados_funcionario)
display(df_funcionarios.head())

In [ ]:
dados_venda = []

STATUS_VENDA = ['APROVADO', 'CANCELADO', 'ESTORNADO']

for i in range(QTD_VENDAS):
    cod_vendedor = randint(1, QTD_FUNCIONARIOS)
    cod_loja = df_funcionarios.loc[
        cod_vendedor - 1, 'cod_loja'
    ]
    
    dados_venda.append({
        'valor_total': round(triangular(0, 999.99, 100), 2),
        'status_venda': choices(STATUS_VENDA, weights=[85, 5, 10], k=1)[0],
        'cod_vendedor': cod_vendedor,
        'cod_loja': cod_loja
    })

df_vendas = pd.DataFrame(dados_venda)
display(df_vendas.head())

In [ ]:
dados_transportadora = []
for i in range(QTD_TRANSPORTADORAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_transportadora.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'telefone': telefone_formatado,
        'email': faker.email(),
        'taxa_entrega': round(uniform(0.01, 1), 2)
    })

df_transportadoras = pd.DataFrame(dados_transportadora)
display(df_transportadoras.head())

In [ ]:
dados_envia_itens = []
dados_conta_loja = []
dados_conta_funcionario = []
dados_pagamento_parcelado = []

dados_conta_loja = [{'cod_loja': i+1, 'cod_conta': randint(1, QTD_CONTAS)} for i in range(QTD_LOJAS)]
for i in range(QTD_CONTA_LOJA - QTD_LOJAS):
    dados_conta_loja.append({'cod_loja': randint(1, QTD_LOJAS), 'cod_conta': randint(1, QTD_CONTAS)})

dados_conta_funcionario = [{'cod_funcionario': i+1, 'cod_conta': randint(1, QTD_CONTAS)} for i in range(QTD_FUNCIONARIOS)]
for i in range(QTD_CONTA_FUNCIONARIO - QTD_FUNCIONARIOS):
    dados_conta_funcionario.append({'cod_funcionario': randint(1, QTD_FUNCIONARIOS), 'cod_conta': randint(1, QTD_CONTAS)})

dados_envia_itens = [
    {'cod_loja': l, 'cod_transportadora': t}
    for l in range(1, QTD_LOJAS + 1)
    for t in range(1, QTD_TRANSPORTADORAS + 1)
]

for i in range(QTD_PAGAMENTO_PARCELADO):
    dados_pagamento_parcelado.append({
        'cod_pagamento': randint(1, QTD_PAGAMENTOS),
        'cod_venda': randint(1, QTD_VENDAS)
    })

df_envia_itens = pd.DataFrame(dados_envia_itens)
df_conta_loja = pd.DataFrame(dados_conta_loja)
df_conta_funcionario = pd.DataFrame(dados_conta_funcionario)
df_pagamento_parcelado = pd.DataFrame(dados_pagamento_parcelado)

In [ ]:
tabelas = {
    'tb_loja': df_lojas,
    'tb_banco': df_bancos,
    'tb_transportadora': df_transportadoras,
    'tb_funcionario': df_funcionarios,
    'tb_conta': df_contas,
    'tb_pagamento': df_pagamentos,
    'tb_venda': df_vendas,
    'tb_envia_itens': df_envia_itens,
    'tb_conta_loja': df_conta_loja,
    'tb_conta_funcionario': df_conta_funcionario,
    'tb_pagamento_parcelado': df_pagamento_parcelado
}

print('Iniciando conexão com o banco de dados')
for t, df in tabelas.items():
    try:
        df.to_sql(name=t, con=engine, if_exists='append', index=False)
    except Exception as e:
        print(f'Um erro no banco aconteceu')
        print(e)
print('Conexão feita com sucesso')